# nb

> Learn with code in a notebook (leela or Jupyter): question cards, and code exercises that the kernel checks.

This module brings the tutor into a notebook. `learn` shows a question card in the output of a cell.
`exercise` gives a code exercise, and its tests run on your function in the kernel.
Both use the same `Tutor` and the same `learning` shelf as the web app.

In [ ]:
#| default_exp nb

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # and the settings, with the saved prompts, go to a temporary folder

These are the imports of the module.

In [ ]:
#| export
import html, os, traceback, uuid
from pathlib import Path
from fastcore.utils import *
from fastcore.nbio import new_nb, mk_cell, write_nb
from abhyasa.store import Learning
from abhyasa.model import DEPTHS
from abhyasa.tutor import Tutor, Session
from abhyasa.settings import val
from abhyasa.prompts import prompt
from abhyasa.llm import ask_json, for_task, route_ask
from abhyasa import demo as _demo   # registers the demo hook of the tutor

## Use

Leela shows ipywidgets, so the card is a widget in the output of the cell.
Leela keeps cell metadata that it does not know. Thus the cells from `write_exercise` keep their `metadata.abhyasa`.

```python
from abhyasa.nb import learn, exercise
learn('reciprocal-rank-fusion')      # a question card
ex = exercise('reciprocal-rank-fusion')
def rrf(lists, k=60): ...            # your code, in the next cell
ex.check(rrf)                        # runs the tests, records the attempt
```

## The tutor

`tutor` makes the tutor of the notebook one time. A call with a store or a model makes a new one.

In [ ]:
#| export
_st = AttrDict(tutor=None)

def tutor(learning=None, ask=None) -> Tutor:
    "The tutor of the notebook, made one time: the default vault and `route_ask`, unless you give others."
    if _st.tutor is None or learning is not None or ask is not None:
        _st.tutor = Tutor(learning or Learning(os.getenv('ABHYASA_VAULT') or os.getenv('VISHALAKSHI_VAULT')), ask or route_ask())
    return _st.tutor

def _esc(s): return html.escape(s or '').replace('\n', '<br>')

## The question card

`Card` has a box for the answer, a box for the think-aloud, a confidence slider and four buttons.

In [ ]:
#| export
class Card:
    "A question card widget over a `Session`: answer, think aloud, confidence, hint, and next."
    def __init__(self, sess:Session):
        import ipywidgets as w
        self.s, self.w = sess, w
        self.q = w.HTMLMath(); self.fb = w.HTMLMath()
        self.ans = w.Textarea(placeholder='Your answer', layout=w.Layout(width='100%', height='70px'))
        self.think = w.Textarea(placeholder='Think aloud: how you got there, what you are unsure of', layout=w.Layout(width='100%', height='90px'))
        self.conf = w.IntSlider(50, 0, 100, description='Confidence')
        b = lambda t, f, st='': (x := w.Button(description=t, button_style=st), x.on_click(lambda _: self._do(f)), x)[-1]
        self.buttons = w.HBox([b('Send', self._send, 'primary'), b('Hint', self._hint), b("I don't know", self._idk), b('Next', self._next)])
        self.demo = w.Output()
        self.box = w.VBox([self.q, self.ans, self.think, self.conf, self.buttons, self.fb, self.demo])
        self._next()

    def _do(self, f):
        try: f()
        except Exception as e: self.fb.value = f'<b>Error</b>: {_esc(str(e))}'

    def _show(self, q):
        if q is None: self.q.value, self.fb.value = '<b>Nothing to learn or review right now.</b>', ''; return
        opts = ''.join(f'<li>{i}. {_esc(o)}</li>' for i, o in enumerate(q.options)) if q.get('options') else ''
        self.q.value = (f"<small>{_esc(q.name)} · {DEPTHS[q.depth]} · {_esc(q.why)}</small><p style='font-size:1.2em'>{_esc(q.question)}</p>"
                        + (f'<ul style="list-style:none">{opts}</ul><small>Answer with the number.</small>' if opts else ''))
        self.ans.value = self.think.value = self.fb.value = ''

    def _send(self):
        r = self.s.answer(self.ans.value, self.think.value, self.conf.value)
        self.fb.value = fb_html(r); self._demo(r)
    def _demo(self, r):
        from IPython.display import display
        self.demo.clear_output()
        if (v := demo_view(r, self.s.tutor.learning)):
            with self.demo: display(v)
    def _hint(self): self.fb.value = f'<i>Hint:</i> {_esc(self.s.hint())}'
    def _idk(self): r = self.s.answer(dont_know=True); self.fb.value = fb_html(r); self._demo(r)
    def _next(self): self._show(self.s.next())
    def _ipython_display_(self):
        from IPython.display import display
        display(self.box)

`fb_html` shows a result in the card. `learn` starts a session in a card.

In [ ]:
#| export
def fb_html(r) -> str:
    "A result as HTML."
    if not r.done: return f"<b>Not yet.</b> {_esc(r.feedback)}<br><small>{r.tries_left} more tries.</small>"
    parts = [f"<b>{r.outcome}</b>", _esc(r.feedback), _esc(r.lesson), f"<small>Floor now {DEPTHS[r.floor]}. Press Next.</small>"]
    return '<br><br>'.join(p for p in parts if p)

def learn(focus:str=None, subject:str=None, limit:int=12) -> Card:
    "A question card for a session, on one concept when you name it."
    return Card(Session(tutor(), focus=focus, subject=subject, limit=limit))

After a miss on a concept that has a demo, the card shows the demo under the feedback. `demo_view` gives the demo as a notebook output. It uses `show` from rupaka, and leela draws the output in a sandboxed frame.

In [ ]:
#| export
def demo_view(r, learning):
    "The demo of result `r` as a notebook output, or None."
    from rupaka.render import show
    return show(d.spec) if r.get('demo') and (d := learning.demo(r.demo)) else None

A result with a demo gives an output with the rupaka mime and a full HTML page. A result without one gives nothing.

In [ ]:
from abhyasa.store import temp_learning
from rupaka.spec import gallery
from rupaka.render import MIME
dl = temp_learning(); dl.put('Vowel sandhi', 'Vowels join by rule.', 'sanskrit')
dl.put_demo(gallery()['vowel-sandhi'], 'vowel-sandhi')
mb = demo_view(AttrDict(demo='vowel-sandhi'), dl)._repr_mimebundle_()
assert MIME in mb and '<script' in mb['text/html']
test_eq(demo_view(AttrDict(correct=True), dl), None)

## Code exercises

The model writes a task (a small function to make), a stub and `assert` tests. The tests run on your function in the kernel.
The exercise shows a failing test, but not the fix. A pass at depth 4 (apply) counts as a pass in the learner model.

`code_sp` and `code_prompt` ask the model for one exercise. `code_sp` reads the tutor rules at call time, so a change on the Settings page applies to the next exercise.

In [ ]:
#| export
CODE_RULES = """
You write one small Python exercise that makes the learner apply a concept by implementing one function.
- The function is 5-25 lines for a correct solution, standard library and numpy only.
- Tests are 3-6 plain `assert` lines that call the function by name; include one edge case.
- Each test is one line with plain values: `assert f(a, b) == c`. A test has no `lambda`, `def`, loop, `import` or helper function. If the function takes a function, take a plain value instead (a number, a string, a list or a dict).
- Good test: `assert softmax([0.0, 0.0]) == [0.5, 0.5]`
- Bad test: `assert decode(lambda s: (s[0], s[1:]), ('a', 'b')) == ['a', 'b']`
- Never put the solution in the stub."""

def code_sp() -> str:
    "The system prompt of a code exercise: the tutor rules now, then the rules of an exercise."
    return prompt('tutor') + CODE_RULES

def code_prompt(c):
    return (f"Concept: {c['name']} (subject: {c.get('subject') or 'general'})\nStatement: {c.get('statement', '')}\n"
            f"Misconceptions seen: {'; '.join(c.get('misconceptions') or []) or 'none'}\n\n"
            'Return JSON: {"task": str, "fname": str, "stub": str, "tests": [str], "solution": str}')

The `check` module validates exercise tests and sets their time limits. This module imports that shared code.

In [ ]:
#| export
from abhyasa.check import SAFE_NAMES, SAFE_BUILTINS, SAFE_NODES, TEST_SECONDS, safe_test, validate_exercise, time_limit

`new_exercise` gets one exercise from the model. A walk uses it too. `Exercise` keeps one exercise open on the tutor. `check` grades it with the same tries and lessons as a question.

In [ ]:
#| export
def new_exercise(ask, c) -> AttrDict:
    "A code exercise on concept `c` from the model, checked by `validate_exercise`. A refused reply gets one more try, with the reason."
    p = code_prompt(c)
    try: return validate_exercise(ask_json(for_task(ask, 'exercise'), p, code_sp()))
    except ValueError as e:
        return validate_exercise(ask_json(for_task(ask, 'exercise'), f'{p}\n\nYour last reply was refused: {e}. Write each test again as one `assert` line with plain values only: no `lambda`, `def`, loop or `import`.', code_sp()))

class Exercise:
    "One code exercise on a concept. `check(fn)` runs its tests on your function."
    def __init__(self, tut:Tutor, slug:str):
        c = tut.learning.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        r = new_exercise(tut.ask, c)
        self.t, self.c = tut, c
        self.q = AttrDict(id=uuid.uuid4().hex[:10], concept=slug, name=c.name, statement=c.statement, depth=4, kind='code',
                          question=r.task, answer=r.solution, rubric=r.tests, hints=[], tries=[], session='', why='code exercise')
        self.fname, self.stub, self.tests = r.fname, r.stub, r.tests
        tut.open[self.q.id] = self.q

    def _ipython_display_(self):
        from IPython.display import display, Markdown
        display(Markdown(f"**{self.c.name}** · code exercise\n\n{self.q.question}\n\n```python\n{self.stub}\n```\n\n"
                         f"Write `{self.fname}` in a cell, then run `ex.check({self.fname})`."))

    def run(self, fn):
        "Keep the shared `report` and return `(test, error)` failures; hidden tests use only label and status."
        from abhyasa.check import run_asserts
        self.report = run_asserts(fn, self.fname, self.tests, secs=val('runner.per_test'), total=val('runner.total'))
        fails = []
        for c in self.report.checks:
            if c.status == 'pass': continue
            t = self.tests[c.i]
            text = c.label if c.hidden else t if isinstance(t, str) else t['test']
            error = c.status if c.hidden else c.error.rstrip(': ') + (f'; got {c.got}' if c.got is not None else '')
            fails.append((text, error))
        return fails

    def check(self, fn, confidence:int=70):
        "Grade the shared report. Hidden failures show only their label and status."
        if self.q.id not in self.t.open: return 'This exercise is closed. Make a new one with `exercise(...)`.'
        fails = self.run(fn)
        passed = sum(c.status == 'pass' for c in self.report.checks)
        first = next((c for c in self.report.checks if c.status != 'pass'), None)
        first_error = ''
        if first is not None:
            first_error = (f'{first.label}: {first.status}' if first.hidden else
                           f'This test fails: `{fails[0][0]}` ({fails[0][1]})')
        g = AttrDict(correct=self.report.passed, grade=10 * passed // len(self.report.checks),
                     first_error=first_error, misconception='', feedback='')
        if fails and len(self.q.tries) + 1 < val('checks.tries'):
            self.q.tries.append(dict(answer=f'{len(fails)} failing', first_error=g.first_error, confidence=confidence))
            return f"{passed} of {len(self.report.checks)} tests pass. {g.first_error}"
        if fails: self.q.tries.append(dict(answer=f'{len(fails)} failing', first_error=g.first_error, confidence=confidence))
        r = self.t._close(self.q, g.correct, self.q.tries[0]['confidence'] if fails else confidence, g, teach=bool(fails))
        return fb_text(r)

    def hint(self) -> str: return self.t.hint(self.q.id)

`fb_text` shows a result as text. `exercise` makes an exercise on the tutor of the notebook.

In [ ]:
#| export
def fb_text(r): return '\n\n'.join(o for o in (r.outcome, r.feedback, r.lesson, f'Floor now {DEPTHS[r.floor]}.') if o)

def exercise(slug:str) -> Exercise:
    "A code exercise on concept `slug`."
    return Exercise(tutor(), slug)

## Exercise notebooks

`write_exercise` writes the exercise into tagged cells of a new notebook file, for you to open later.
It never writes to a notebook that is already there.

In [ ]:
#| export
def write_exercise(ex:Exercise, path) -> Path:
    "A new notebook at `path` with the task, a stub cell and the check. Do not write over a file."
    p = Path(path).expanduser()
    if p.exists(): raise FileExistsError(p)
    tag = dict(abhyasa=dict(concept=ex.q.concept, qid=ex.q.id))
    cells = [mk_cell(f"# {ex.c.name}\n\n{ex.q.question}", 'markdown', metadata=tag),
             mk_cell(ex.stub, metadata=tag),
             mk_cell('\n'.join(ex.tests), metadata=dict(abhyasa=dict(concept=ex.q.concept, qid=ex.q.id, tests=True)))]
    write_nb(new_nb(cells), p)
    return p

## Tests

The first tests check the safety rules. Then a fake model writes a full exercise.

In [ ]:
#| hide

In [ ]:
#| hide

In [ ]:
#| hide

A fake model writes an exercise with two tests. The tutor of the notebook uses a temporary shelf.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml')
fake = Fake({'Return JSON: {"task"': dict(task='Write rrf(lists, k) that returns items by fused score.', fname='rrf',
             stub='def rrf(lists, k=60):\n    ...', tests=["assert rrf([['a','b'],['b','a']])[0] in 'ab'", "assert rrf([['a'],['b']], k=0) == ['a','b']"],
             solution='...'), 'Teach the smallest step': 'Sum 1/(k+r). Check: what if k is large?'},
            dflt=dict(question='Why ranks?', answer='Comparable.', rubric=['comparable']))
t = tutor(lr, fake)
ex = exercise('reciprocal-rank-fusion')

A wrong function fails both tests, and the exercise stays open.

In [ ]:
def rrf(lists, k=60): return []
assert ex.check(rrf).startswith('0 of 2 tests pass')

A correct function passes after the failed try, so the outcome is `fragile` and the concept is `shaky` at depth 4.

In [ ]:
def rrf(lists, k=60):
    s = {}
    for l in lists:
        for r, x in enumerate(l): s[x] = s.get(x, 0) + 1 / (k + r + 1)
    return sorted(s, key=lambda x: (-s[x], x))
out = ex.check(rrf)
assert out.startswith('fragile'), out     # passed after a failed try
test_eq(lr.concept('reciprocal-rank-fusion').shaky, 4)

`write_exercise` tags the cells and does not write over a file.

In [ ]:
import tempfile
from fastcore.nbio import read_nb
ex2 = exercise('reciprocal-rank-fusion')
p = write_exercise(ex2, Path(tempfile.mkdtemp())/'rrf.ipynb')
nb = read_nb(p)
test_eq(nb.cells[1].metadata['abhyasa']['concept'], 'reciprocal-rank-fusion')
test_fail(lambda: write_exercise(ex2, p), exc=FileExistsError)

`learn` shows a card with the first question.

In [ ]:
c = learn('reciprocal-rank-fusion')
assert 'Why ranks?' in c.q.value

The default model of the notebook tutor is the routed one. This test swaps `route_ask` for a marker, so it makes no model call.

In [ ]:
real, mark = route_ask, lambda p, sp='': 'ok'
route_ask = lambda: mark
try:
    _st.tutor = None
    assert tutor(lr).ask is mark
finally: route_ask = real
t = tutor(lr, fake)

An exercise asks on the `exercise` task. With no model of its own, it uses the model and the reply cap of the `question` task. This ask notes the task that it gets.

In [ ]:
class Routed:
    def __init__(s, f): s.f, s.tasks = f, []
    def for_(s, task): s.tasks.append(task); return s.f
r = Routed(fake)
tutor(lr, r); exercise('reciprocal-rank-fusion')
test_eq(r.tasks, ['exercise'])
t = tutor(lr, fake)

A saved "Tutor rules" text reaches the code prompt on the next call, with no new import. An empty text brings the default back.

In [ ]:
from abhyasa.prompts import set_prompt
set_prompt('tutor', 'Rules of the test. Be brief.')
assert code_sp().startswith('Rules of the test.') and 'one small Python exercise' in code_sp()
set_prompt('tutor', '')
assert code_sp().startswith('You are a tutor')

A test with a `lambda` is refused. `new_exercise` tries again with the reason, and the second reply passes. The rules also show a good test and a bad one.


In [ ]:
from abhyasa.llm import Fake
good = dict(task='Add two numbers.', fname='add', stub='def add(a, b):\n    ...', tests=['assert add(1, 2) == 3', 'assert add(0, 0) == 0', 'assert add(-1, 1) == 0'], solution='def add(a, b):\n    return a + b')
bad = dict(good, tests=['assert add(lambda: 1, 2) == 3', 'assert add(0, 0) == 0', 'assert add(-1, 1) == 0'])
replies = iter([bad, good])
fake = Fake(dflt=lambda p, sp: next(replies))
c = dict(name='Adding', slug='adding', statement='Add two numbers.')
test_eq(new_exercise(fake, c).fname, 'add')
test_eq(len(fake.calls), 2)
assert 'refused' in fake.calls[1][0] and 'no `lambda`' in fake.calls[1][0]    # the second try gets the reason
assert 'Bad test:' in code_sp() and 'Good test:' in code_sp()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()